In [2]:
import tensorflow as tf
import keras
from keras import layers as tfkl
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
dataset = np.concatenate([x_train,x_test])
dataset = dataset/255.0
dataset = np.expand_dims(dataset , axis = -1)
print(dataset.shape)

(70000, 28, 28, 1)


In [3]:
discriminator_net = keras.Sequential(
    [
        keras.Input(shape = (28,28,1)),
        tfkl.Conv2D(filters = 64 , kernel_size = 4 , strides = 2 , padding = 'same'),
        tfkl.LeakyReLU(alpha=0.2),
        tfkl.Conv2D(filters = 128 , kernel_size = 4 , strides = 2 , padding = 'same'),
        tfkl.LeakyReLU(alpha=0.2),
        tfkl.Conv2D(filters = 128 , kernel_size = 4 , strides = 2 , padding = 'same'),
        tfkl.LeakyReLU(alpha=0.2),
        tfkl.Flatten(),
        tfkl.Dropout(0.2),
        tfkl.Dense(1 , activation = 'sigmoid')
    ],
    name = 'Discriminator',
)
discriminator_net.summary()

Model: "Discriminator"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 14, 14, 64)        1088      
                                                                 
 leaky_re_lu (LeakyReLU)     (None, 14, 14, 64)        0         
                                                                 
 conv2d_1 (Conv2D)           (None, 7, 7, 128)         131200    
                                                                 
 leaky_re_lu_1 (LeakyReLU)   (None, 7, 7, 128)         0         
                                                                 
 conv2d_2 (Conv2D)           (None, 4, 4, 128)         262272    
                                                                 
 leaky_re_lu_2 (LeakyReLU)   (None, 4, 4, 128)         0         
                                                                 
 flatten (Flatten)           (None, 2048)            

In [4]:
latent_dim = 20

generator = keras.Sequential(
    [
        keras.Input(shape=(latent_dim,)),
        tfkl.Dense(3 * 3 * 128),
        tfkl.Reshape((3, 3, 128)),
        tfkl.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same"),
        tfkl.LeakyReLU(alpha=0.2),
        tfkl.Conv2DTranspose(256, kernel_size=4, strides=2, padding="valid"),
        tfkl.LeakyReLU(alpha=0.2),
        tfkl.Conv2DTranspose(512, kernel_size=4, strides=2, padding="same"),
        tfkl.LeakyReLU(alpha=0.2),
        tfkl.Conv2D(1, kernel_size=5, padding="same", activation="sigmoid"),
    ],
    name="generator",
)
generator.summary()

Model: "generator"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_1 (Dense)             (None, 1152)              24192     
                                                                 
 reshape (Reshape)           (None, 3, 3, 128)         0         
                                                                 
 conv2d_transpose (Conv2DTra  (None, 6, 6, 128)        262272    
 nspose)                                                         
                                                                 
 leaky_re_lu_3 (LeakyReLU)   (None, 6, 6, 128)         0         
                                                                 
 conv2d_transpose_1 (Conv2DT  (None, 14, 14, 256)      524544    
 ranspose)                                                       
                                                                 
 leaky_re_lu_4 (LeakyReLU)   (None, 14, 14, 256)       0 

In [5]:

class GAN(keras.Model):
    def __init__(self, discriminator, generator, latent_dim):
        super().__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        self.seed_generator = 1337

    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super().compile()
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.loss_fn = loss_fn
        self.d_loss_metric = keras.metrics.Mean(name="d_loss")
        self.g_loss_metric = keras.metrics.Mean(name="g_loss")

    @property
    def metrics(self):
        return [self.d_loss_metric, self.g_loss_metric]

    def train_step(self, real_images):
        # Sample random points in the latent space
        batch_size = tf.shape(real_images)[0]
        random_latent_vectors = tf.random.normal(
            shape=(batch_size, self.latent_dim), seed=self.seed_generator, dtype=tf.float32
        )

        # Decode them to fake images
        generated_images = self.generator(random_latent_vectors)

        # Combine them with real images
        combined_images = tf.concat([generated_images, real_images], axis=0)

        # Assemble labels discriminating real from fake images
        labels = tf.concat(
            [tf.ones((batch_size, 1)), tf.zeros((batch_size, 1))], axis=0
        )
        # Add random noise to the labels
        labels += 0.05 * tf.random.uniform(tf.shape(labels))

        # Train the discriminator
        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(
            zip(grads, self.discriminator.trainable_weights)
        )

        # Sample random points in the latent space
        random_latent_vectors = tf.random.normal(
            shape=(batch_size, self.latent_dim), seed=self.seed_generator
        )

        # Assemble labels that say "all real images"
        misleading_labels = tf.zeros((batch_size, 1))

        # Train the generator (note that we should *not* update the weights
        # of the discriminator)!
        with tf.GradientTape() as tape:
            predictions = self.discriminator(self.generator(random_latent_vectors))
            g_loss = self.loss_fn(misleading_labels, predictions)
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        # Update metrics
        self.d_loss_metric.update_state(d_loss)
        self.g_loss_metric.update_state(g_loss)
        return {
            "d_loss": self.d_loss_metric.result(),
            "g_loss": self.g_loss_metric.result(),
        }

In [3]:
class GANMonitor(keras.callbacks.Callback):
    def __init__(self, num_img=3, latent_dim=128):
        self.num_img = num_img
        self.latent_dim = latent_dim
        self.seed_generator = 42

    def on_epoch_end(self, epoch, logs=None):
        random_latent_vectors = tf.random.normal(
            shape=(self.num_img, self.latent_dim), seed=self.seed_generator
        )
        generated_images = self.model.generator(random_latent_vectors)
        generated_images *= 255
        generated_images.numpy()
        for i in range(self.num_img):
            img = keras.utils.array_to_img(generated_images[i])
            img.save("generated_img_%03d_%d.png" % (epoch, i))

In [ ]:
epochs = 10

gan = GAN(discriminator=discriminator_net, generator=generator, latent_dim=latent_dim)
gan.compile(
    d_optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    g_optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss_fn=keras.losses.BinaryCrossentropy(),
)

gan.fit(
    dataset, epochs=epochs, callbacks=[GANMonitor(num_img=10, latent_dim=latent_dim)]
)